# Sentiment Analysis

This notebook covers Sentiment Analysis, one of the most common and practical applications of natural language processing. Sentiment analysis involves determining the emotional tone or opinion expressed in text, typically classifying it as positive, negative, or neutral.

This technique is widely used for monitoring customer feedback, social media analysis, brand reputation management, and market research.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

## 1. Understanding Sentiment Analysis

Sentiment analysis (or opinion mining) is the process of determining the emotional tone behind text. It can be performed at different levels:

- **Document-level**: Classifying the sentiment of an entire document
- **Sentence-level**: Classifying the sentiment of each sentence
- **Aspect-level**: Analysing sentiment towards specific aspects of a product/service

Common approaches include:
- Rule-based methods (using sentiment lexicons)
- Machine learning approaches (traditional classifiers)
- Deep learning (LSTMs, transformers)
- Pre-trained language models (BERT, etc.)

In [ ]:
# Create a sample dataset for sentiment analysis
# In practice, you'd use datasets like IMDb, Yelp, or Twitter data

data = [
    ("This product is amazing! I love it so much.", "positive"),
    ("Absolutely fantastic, highly recommend!", "positive"),
    ("Great quality and fast delivery.", "positive"),
    ("Best purchase I've ever made.", "positive"),
    ("Wonderful experience, will buy again.", "positive"),
    ("Very satisfied with this item.", "positive"),
    ("Excellent service and friendly staff.", "positive"),
    ("The food was delicious and fresh.", "positive"),
    ("Terrible product, complete waste of money.", "negative"),
    ("Very disappointed with this purchase.", "negative"),
    ("Worst experience ever, do not buy.", "negative"),
    ("Poor quality, broke after one use.", "negative"),
    ("Horrible customer service, never again.", "negative"),
    ("The product didn't work as advertised.", "negative"),
    ("Slow delivery and damaged package.", "negative"),
    ("Tasteless food, will not return.", "negative"),
    ("The weather is okay today.", "neutral"),
    ("It's a regular day, nothing special.", "neutral"),
    ("The meeting is scheduled for 3pm.", "neutral"),
    ("I received the package on time.", "neutral"),
    ("The book has 200 pages.", "neutral"),
    ("Store opens at 9am.", "neutral"),
    ("The address is correct.", "neutral"),
    ("It costs £20.", "neutral"),
]

# Convert to DataFrame
df = pd.DataFrame(data, columns=['text', 'sentiment'])

print("=== Sample Dataset ===")
print(f"Total samples: {len(df)}")
print(f"\nClass distribution:")
print(df['sentiment'].value_counts())
print(f"\nSample data:")
print(df.head(10))

In [ ]:
# Display sample texts for each sentiment
print("=== Sample Texts by Sentiment ===\n")

for sentiment in ['positive', 'negative', 'neutral']:
    print(f"{sentiment.upper()}:")
    samples = df[df['sentiment'] == sentiment]['text'].head(3).tolist()
    for s in samples:
        print(f"  - {s}")
    print()

## 2. Building a Sentiment Classifier

We'll build a sentiment classifier using TF-IDF features and Logistic Regression. This is a classic and effective approach for sentiment analysis.

In [ ]:
# Prepare features and labels
X = df['text']
y = df['sentiment']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== Data Split ===")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Create TF-IDF features
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),  # Unigrams and bigrams
    stop_words='english'
)

# Fit and transform training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("=== TF-IDF Features ===")
print(f"Training feature shape: {X_train_tfidf.shape}")
print(f"Test feature shape: {X_test_tfidf.shape}")
print(f"\nFeature names (sample): {tfidf_vectorizer.get_feature_names_out()[:10]}")

In [ ]:
# Train a Logistic Regression classifier
classifier = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial'
)

classifier.fit(X_train_tfidf, y_train)

print("=== Model Training ===")
print("Logistic Regression classifier trained successfully.")

In [ ]:
# Make predictions on test set
y_pred = classifier.predict(X_test_tfidf)

print("=== Model Evaluation ===")
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("=== Confusion Matrix ===")
print(pd.DataFrame(
    cm,
    index=['Actual: ' + s for s in classifier.classes_],
    columns=['Pred: ' + s for s in classifier.classes_]
))

## 3. Testing the Classifier

Let's test our classifier on new, unseen text.

In [ ]:
# Test on new sentences
test_sentences = [
    "This is absolutely brilliant! Best ever!",
    "I hate this, it's the worst thing ever.",
    "The meeting starts at 2pm in conference room B.",
    "So happy with my purchase, works perfectly!",
    "Very poor quality, would not recommend.",
    "The product is exactly as described.",
]

print("=== Testing on New Sentences ===\n")

# Transform test sentences
test_tfidf = tfidf_vectorizer.transform(test_sentences)

# Predict
predictions = classifier.predict(test_tfidf)
probabilities = classifier.predict_proba(test_tfidf)

for i, (sent, pred) in enumerate(zip(test_sentences, predictions)):
    print(f"Text: {sent}")
    print(f"Predicted sentiment: {pred}")
    print(f"Probabilities: {dict(zip(classifier.classes_, probabilities[i]))}")
    print()

## 4. Using Pre-trained Transformers for Sentiment Analysis

For better accuracy, we can use pre-trained transformer models like BERT. The transformers library provides easy-to-use sentiment analysis pipelines.

In [ ]:
from transformers import pipeline

# Load pre-trained sentiment analysis model
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("=== Pre-trained Sentiment Model ===")
print("Model: distilbert-base-uncased-finetuned-sst-2-english")
print("This model is fine-tuned on the Stanford Sentiment Treebank (SST-2).")

In [ ]:
# Test with the same sentences
print("=== Testing Pre-trained Model ===\n")

for sent in test_sentences:
    result = sentiment_analyzer(sent)[0]
    print(f"Text: {sent}")
    print(f"Sentiment: {result['label']} (confidence: {result['score']:.4f})")
    print()

## 5. Sentiment Analysis with VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment analyser specifically tuned for social media text. It's part of NLTK.

In [ ]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Initialise VADER
vader = SentimentIntensityAnalyzer()

print("=== VADER Sentiment Analysis ===")
print("VADER is specifically tuned for social media text.")
print()

# Test with sample sentences
for sent in test_sentences:
    scores = vader.polarity_scores(sent)
    
    # Determine sentiment based on compound score
    if scores['compound'] >= 0.05:
        sentiment = "positive"
    elif scores['compound'] <= -0.05:
        sentiment = "negative"
    else:
        sentiment = "neutral"
    
    print(f"Text: {sent}")
    print(f"Sentiment: {sentiment} (compound: {scores['compound']:.4f})")
    print()

## Summary

In this notebook, we covered Sentiment Analysis:

1. **Understanding**: Learned about different levels of sentiment analysis
2. **Dataset**: Created a sample dataset with positive, negative, and neutral text
3. **TF-IDF + Logistic Regression**: Built a classic ML sentiment classifier
4. **Evaluation**: Used accuracy, precision, recall, and F1-score
5. **Pre-trained Transformers**: Used BERT-based model for better accuracy
6. **VADER**: Explored rule-based sentiment analysis for social media

Sentiment analysis is a fundamental NLP task with many real-world applications. The approach you choose depends on your specific needs:
- Rule-based (VADER) for speed and simplicity
- Traditional ML (TF-IDF + LR) for balance of speed and accuracy
- Transformers for highest accuracy on complex text